I implemented the XGBoost model and carried out hyperparameter tuning using RandomizedSearchCV.
XGBoost combines boosting and regularization to enhance predictive accuracy and reduce bias.
After optimization, our best parameters included 400 estimators, max depth of 4, and a learning rate of 0.1.
It achieved the highest standalone accuracy of about 74.2% with a macro F1-score of 0.742.
This model became our best individual performer, effectively identifying subtle differences between mental health categories.

In [1]:
from google.colab import files
uploaded = files.upload()

Saving preprocessed_data.csv to preprocessed_data.csv


In [2]:
import pandas as pd, numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

df = pd.read_csv("preprocessed_data.csv").dropna(subset=["processed_text","target"]).reset_index(drop=True)
X, y = df["processed_text"].tolist(), df["target"]

model = SentenceTransformer("all-MiniLM-L6-v2")
X_emb = model.encode(X, batch_size=64, show_progress_bar=True)

X_train, X_test, y_train, y_test = train_test_split(X_emb, y, test_size=0.2, random_state=42, stratify=y)

xgb = XGBClassifier(objective="multi:softprob", num_class=5, random_state=42, n_jobs=-1)

param_dist = {
    "n_estimators": [200, 400, 600],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

search = RandomizedSearchCV(
    xgb, param_distributions=param_dist,
    n_iter=10, cv=3, scoring="f1_macro", n_jobs=-1, verbose=1, random_state=42
)

search.fit(X_train, y_train)
best_xgb = search.best_estimator_

y_pred = best_xgb.predict(X_test)

print("=== Tuned XGBoost ===")
print("Best params:", search.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Fitting 3 folds for each of 10 candidates, totalling 30 fits
=== Tuned XGBoost ===
Best params: {'subsample': 0.8, 'n_estimators': 600, 'max_depth': 6, 'learning_rate': 0.1, 'colsample_bytree': 0.8}
Accuracy: 0.7507418397626113
              precision    recall  f1-score   support

           0       0.84      0.77      0.80       189
           1       0.63      0.66      0.65       187
           2       0.81      0.77      0.79       203
           3       0.73      0.74      0.73       208
           4       0.76      0.80      0.78       224

    accuracy                           0.75      1011
   macro avg       0.75      0.75      0.75      1011
weighted avg       0.75      0.75      0.75      1011

[[145  14   2   9  19]
 [  8 124  15  25  15]
 [  3  17 157  11  15]
 [  5  34   7 153   9]
 [ 11   7  14  12 180]]
